In [1]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from tqdm import tqdm
from scipy.stats import (skew, 
                        kurtosis)
from scipy.signal import welch
from scipy.signal import hilbert
import pywt

# Load the features

In [2]:
data = pd.read_pickle("../data/processed/eeg_bandpass_ica.pkl")
data = data[['subject', 'trial', 'trail_type', 'label', 'label_description', 'shape', 'epoch_data_ICA', 'epoch_data_ICA_inds']]
print(f"Shape of the dataset : {data.shape}")

Shape of the dataset : (4890, 8)


In [3]:
eeg_data_cleaned = data["epoch_data_ICA"].tolist()

# Extract Time-Related Features

#### Step 1: Extract Time features

In [4]:
def extract_time_features(X):
    """Extract time-domain features"""
    n_channels, n_samples = X.shape
    
    # Start with empty DataFrame
    features_df = pd.DataFrame()

    for ch in range(n_channels):
        signal = X[ch, :]
        
        # Statistical features
        trial_features = [
            np.mean(signal),          # Mean
            np.std(signal),           # Standard deviation
            np.var(signal),           # Variance
            skew(signal),             # Skewness
            kurtosis(signal),         # Kurtosis
            np.max(signal),           # Maximum
            np.min(signal),           # Minimum
            np.ptp(signal),           # Peak-to-peak
            np.median(signal),        # Median
            np.mean(np.abs(signal)),  # Mean absolute value
            np.sum(np.diff(np.sign(signal)) != 0),  # Zero crossing rate
            np.sqrt(np.mean(signal**2)),            # Root mean square
        ]
        
        feature_names = ['mean', 'std', 'var', 'skew', 'kurt', 'max', 'min', 'ptp', 'median', 'mean_abs', 'zero_cross', 'rms']
        columns = [f"ch_{ch+1}_{feat}" for feat in feature_names]
        
         # Create dataframe for this channel
        channel_df = pd.DataFrame([trial_features], columns=columns)

        # Append column-wise
        features_df = pd.concat([features_df, channel_df], axis=1)
    
    return features_df

In [5]:
X_train_features = [extract_time_features(eeg_data_cleaned[i]) for i in tqdm(range(0, len(eeg_data_cleaned)))]
X_train_features = pd.concat(X_train_features)

# Make sure the index is correct
X_train_features.index = data.index 

# Append X_train_features to data column-wise
X_train_features = pd.concat([data, X_train_features], axis= 1)

100%|██████████| 4890/4890 [02:36<00:00, 31.16it/s]


#### Step 2: Save the dataframe

In [6]:
X_train_features.to_pickle("../data/feature_mart/train_features.pkl")

# Extract Frequency-Related Features

In [7]:
def extract_frequency_features(X, sfreq, freq_bands):
        """Extract frequency-domain features"""
        n_channels, n_samples = X.shape
        
        features_df = pd.DataFrame()
            
        for ch in range(n_channels):

            signal = X[ch, :]
            
            # Power spectral density
            freqs, psd = welch(signal, fs=sfreq, nperseg=min(256, n_samples//4))
            

            trial_features = []
            # Band power features
            for band_name, (low, high) in freq_bands.items():
                band_mask = (freqs >= low) & (freqs <= high)
                band_power = np.sum(psd[band_mask])
                trial_features.append(band_power)
            
            # Spectral features
            total_power = np.sum(psd)
            spectral_centroid = np.sum(freqs * psd) / total_power
            spectral_spread = np.sqrt(np.sum(((freqs - spectral_centroid)**2) * psd) / total_power)
            spectral_entropy = -np.sum((psd/total_power) * np.log2(psd/total_power + 1e-12))
            
            trial_features.extend([
                total_power,
                spectral_centroid,
                spectral_spread,
                spectral_entropy
            ])
            
            # Peak frequency
            peak_freq = freqs[np.argmax(psd)]
            trial_features.append(peak_freq)

            # Create dataframe for this channel
            freq_features_names = ['delta', 'theta', 'alpha', 'beta', 'gamma', 'total_power', 'centroid', 'spread', 'entropy', 'peak_freq']
            columns = [f"ch_{ch+1}_{feat}" for feat in freq_features_names]
            channel_df = pd.DataFrame([trial_features], columns=columns)
            
            # Append column-wise
            features_df = pd.concat([features_df, channel_df], axis=1)

            
        return features_df

In [8]:
freq_bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13),
 'beta': (13, 30), 'gamma': (30, 50)}
sfreq = 250


X_frequence_features = [extract_frequency_features(X = eeg_data_cleaned[i], 
                                                  sfreq = 250, 
                                                  freq_bands= freq_bands) for i in tqdm(range(0, len(eeg_data_cleaned)))]
X_frequence_features = pd.concat(X_frequence_features)

# Make sure the index is correct
X_frequence_features.index = data.index 

# Append X_train_features to data column-wise
X_train_freq_features = pd.concat([data, X_frequence_features], axis= 1)

100%|██████████| 4890/4890 [01:34<00:00, 51.70it/s]


#### Step 2: Save the features

In [9]:
X_train_freq_features.to_pickle("../data/feature_mart/train_frequency_features.pkl")

# Extract Time-Frequency Related Features

In [11]:
def extract_joint_features(X, sfreq, wavelet="morl", max_scale=30):
    """
    Extract time-frequency joint features for each channel in one trial.
    Returns a single-row DataFrame (features flattened column-wise).
    
    Parameters
    ----------
    X : ndarray, shape (n_channels, n_samples)
        EEG trial data.
    sfreq : float
        Sampling frequency.
    wavelet : str
        Wavelet type (default: 'morl' - Morlet).
    max_scale : int
        Maximum scale for CWT.

    Returns
    -------
    features_df : pandas.DataFrame
        Single-row DataFrame with features per channel flattened column-wise.
    """
    n_channels, n_samples = X.shape
    
    feature_names = [
        "amp_mean", "amp_std", "amp_max", "amp_skew",
        "freq_mean", "freq_std", "freq_median",
        "wavelet_energy_1_5", "wavelet_energy_6_15", "wavelet_energy_16_30"
    ]
    
    all_features = {}
    
    for ch in range(n_channels):
        signal = X[ch, :]
        trial_features = []
        
        # --- Hilbert transform ---
        analytic_signal = hilbert(signal)
        amplitude_envelope = np.abs(analytic_signal)
        instantaneous_phase = np.unwrap(np.angle(analytic_signal))
        instantaneous_freq = np.diff(instantaneous_phase) / (2.0 * np.pi) * sfreq
        
        # Amplitude envelope features
        trial_features.extend([
            np.mean(amplitude_envelope),
            np.std(amplitude_envelope),
            np.max(amplitude_envelope),
            skew(amplitude_envelope)
        ])
        
        # Instantaneous frequency features
        if len(instantaneous_freq) > 0:
            trial_features.extend([
                np.mean(instantaneous_freq),
                np.std(instantaneous_freq),
                np.median(instantaneous_freq)
            ])
        else:
            trial_features.extend([0, 0, 0])
        
        # --- Wavelet features using PyWavelets CWT ---
        scales = np.arange(1, max_scale + 1)
        coefficients, freqs = pywt.cwt(signal, scales, wavelet, 1.0/sfreq)
        
        for scale_group in [(1, 5), (6, 15), (16, 30)]:
            scale_energy = np.sum(np.abs(coefficients[scale_group[0]-1:scale_group[1]])**2)
            trial_features.append(scale_energy)
        
        # Store with channel-specific names
        for fname, fval in zip(feature_names, trial_features):
            all_features[f"{fname}_ch{ch}"] = fval
    
    # Return as single-row DataFrame
    features_df = pd.DataFrame([all_features])
    
    return features_df


In [12]:
X_joint_features = [extract_joint_features(X = eeg_data_cleaned[i], 
                                                  sfreq = 250, 
                                                ) for i in tqdm(range(0, len(eeg_data_cleaned)))]
X_joint_features = pd.concat(X_joint_features)

# Make sure the index is correct
X_joint_features.index = data.index 

# Append X_train_features to data column-wise
X_joint_features = pd.concat([data, X_joint_features], axis= 1)

100%|██████████| 4890/4890 [06:09<00:00, 13.22it/s]


In [13]:
X_joint_features.to_pickle("../data/feature_mart/train_join_features.pkl")

# Extract CSP related Features